# VELSIPITY Dataset: Production Cleaning & Feature Reduction Pipeline

**Dataset:** `VELSIPITY_AFF_MULTIPLI_prepared_prepared_prepared`  
**Domain:** Pharma Commercial Analytics  
**Grain:** `NUEVO_ID × WEEK_ID` (weekly panel)  
**Prepared by:** Senior Data Science Pipeline  

---
## Notebook Structure
1. Setup & Data Loading
2. Structural Validation
3. Data Cleaning (Missing, Types, Outliers)
4. Feature Reduction (Redundancy, Multicollinearity, Leakage)
5. Temporal Integrity & Lag Features
6. Final Dataset Construction & Export
7. Summary Report


---
## 1. Setup & Data Loading

In [28]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Config ──────────────────────────────────────────────────────────────────
INPUT_FILE  = 'VELSIPITY_AFF_MULTIPLI_prepared_prepared_prepared.csv'
OUTPUT_FILE = 'VELSIPITY_cleaned_igna.csv'
RANDOM_SEED = 42

# Load full dataset
df_raw = pd.read_csv(INPUT_FILE, low_memory=False)

print(f'Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Memory: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_raw.head(3)

Loaded: 1,800,066 rows × 68 columns
Memory: 1159.3 MB


,NUEVO_ID,WEEK_ID,UC_TRX,ORAL_TRX,IL23_TRX,BRAND1_TRX,BRAND2_TRX,UC_NRX,ORAL_NRX,IL23_NRX,...,STATE_6,STS_OTHER_STS,STATE_7,STATE_8,"(1940, 1960]","(1960, 1980]","(1980, 2000]","(2000, 2020]","(2020, 2030]",ATSEG
0,17962,2024-08-02,0.1652,0.0000,0.0,0.0,0.0,0.0000,0.0000,0.0,...,0,0,0,0,0,0,0,0,0,NaN
1,3802,2024-11-08,1.5024,0.3780,0.0,0.0,0.0,0.0000,0.0000,0.0,...,0,1,0,0,0,1,0,0,0,SEG_C
2,422,2025-04-25,0.3558,0.1906,0.0,0.0,0.0,0.5337,0.2859,0.0,...,0,0,0,0,1,0,0,0,0,SEG_C


---
## 2. Structural Validation
Confirm panel grain, completeness, and key integrity.

In [29]:
# ── 2.1 Panel grain check ───────────────────────────────────────────────────
n_ids   = df_raw['NUEVO_ID'].nunique()
n_weeks = df_raw['WEEK_ID'].nunique()
expected_rows = n_ids * n_weeks
observed_rows = len(df_raw)

print('=== Panel Completeness ===')
print(f'  Unique NUEVO_ID : {n_ids:,}')
print(f'  Unique WEEK_ID  : {n_weeks}')
print(f'  Expected rows   : {expected_rows:,}')
print(f'  Observed rows   : {observed_rows:,}')
status = 'COMPLETE ✓' if expected_rows == observed_rows else 'INCOMPLETE ✗'
print(f'  Status          : {status}')

# ── 2.2 Duplicate check ─────────────────────────────────────────────────────
n_dupes = df_raw.duplicated(subset=['NUEVO_ID','WEEK_ID']).sum()
print(f'\n  Duplicate ID×WEEK rows: {n_dupes}')
assert n_dupes == 0, 'Duplicates found — investigate before proceeding!'

=== Panel Completeness ===
  Unique NUEVO_ID : 20,931
  Unique WEEK_ID  : 86
  Expected rows   : 1,800,066
  Observed rows   : 1,800,066
  Status          : COMPLETE ✓

  Duplicate ID×WEEK rows: 0


In [30]:
# ── 2.3 Column inventory ────────────────────────────────────────────────────
print('=== Column Inventory ===')
print(f'Total columns: {df_raw.shape[1]}')
print()
print('Data types:')
print(df_raw.dtypes.value_counts())

=== Column Inventory ===
Total columns: 68

Data types:
float64    42
int64      24
object      2
Name: count, dtype: int64


---
## 3. Data Cleaning
### 3.1 Temporal Field Casting

In [31]:
df = df_raw.copy()

# ── Cast WEEK_ID to datetime (ISO format: YYYY-MM-DD) ─────────────────────
# Rationale: WEEK_ID is the primary time axis for the panel model.
# Proper datetime type enables lag creation, sorting, and temporal splits.
df['WEEK_ID'] = pd.to_datetime(df['WEEK_ID'], errors='coerce')

# Sanity check: no null weeks after cast
null_weeks = df['WEEK_ID'].isnull().sum()
print(f'Null WEEK_ID after cast: {null_weeks}')
assert null_weeks == 0, 'Some WEEK_ID values could not be parsed!'

# YEAR and QTR are redundant with WEEK_ID — we'll revisit in Section 4.
# YEAR_QTR is also derivable from WEEK_ID.
print('WEEK_ID range:', df['WEEK_ID'].min().date(), '→', df['WEEK_ID'].max().date())
print('Sample WEEK_ID values:', df['WEEK_ID'].unique()[:5])

Null WEEK_ID after cast: 0
WEEK_ID range: 2024-01-05 → 2025-08-22
Sample WEEK_ID values: <DatetimeArray>
['2024-08-02 00:00:00', '2024-11-08 00:00:00', '2025-04-25 00:00:00',
 '2024-02-16 00:00:00', '2024-01-12 00:00:00']
Length: 5, dtype: datetime64[ns]


### 3.2 Missing Value Standardization

In [32]:
# ── 3.2.1 ATSEG: Normalize all missing representations ─────────────────────
# Known issues: null, empty string '', literal 'missing value'
# Decision: unify all to NaN, then handle downstream.

atseg_before = df['ATSEG'].value_counts(dropna=False)

df['ATSEG'] = df['ATSEG'].replace({'missing value': np.nan, '': np.nan})
df['ATSEG'] = df['ATSEG'].where(df['ATSEG'].notna(), np.nan)

atseg_after = df['ATSEG'].value_counts(dropna=False)

print('=== ATSEG Before ===')
print(atseg_before)
print('\n=== ATSEG After Normalization ===')
print(atseg_after)
print(f'\nMissing rate: {df["ATSEG"].isnull().mean():.1%}')

=== ATSEG Before ===
ATSEG
NaN      776752
SEG_A    550916
SEG_B    288014
SEG_C    184384
Name: count, dtype: int64

=== ATSEG After Normalization ===
ATSEG
NaN      776752
SEG_A    550916
SEG_B    288014
SEG_C    184384
Name: count, dtype: int64

Missing rate: 43.2%


In [33]:
# ── 3.2.2 Add ATSEG missingness flag ───────────────────────────────────────
# Rationale: With ~43% missingness, ATSEG cannot be imputed reliably.
# We add a binary flag to preserve the signal that missingness itself carries
# (missing ATSEG may indicate a specific provider segment or data pipeline gap).
# The original NaN values are kept; models should handle missing ATSEG explicitly.

df['ATSEG_IS_MISSING'] = df['ATSEG'].isnull().astype(int)
print('ATSEG_IS_MISSING distribution:')
print(df['ATSEG_IS_MISSING'].value_counts())

ATSEG_IS_MISSING distribution:
ATSEG_IS_MISSING
0    1023314
1     776752
Name: count, dtype: int64


In [34]:
# ── 3.2.3 Overall missingness audit ────────────────────────────────────────
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
print('Columns with missing values:')
if len(missing_summary) == 0:
    print('  None (except ATSEG handled above)')
else:
    print(missing_summary)

Columns with missing values:
ATSEG    776752
dtype: int64


### 3.3 Zero-Inflation Assessment

In [35]:
# ── Identify zero-heavy numeric columns ─────────────────────────────────────
# Rationale: In weekly pharma panels, many activity variables (prescriptions,
# claims, promotions) are zero for most HCPs in any given week. This is
# structurally expected and does NOT require imputation or special treatment
# per se — but extremely zero-heavy columns (>99%) provide near-zero variance
# and may be candidates for binary transformation or removal.

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['NUEVO_ID','YEAR','QTR','YEAR_QTR']]

zero_rate = (df[numeric_cols] == 0).mean().sort_values(ascending=False)

print('=== Zero Inflation by Column ===')
print(zero_rate.to_string())

# Flag columns that are >99% zero (extremely sparse)
ultra_sparse = zero_rate[zero_rate > 0.99].index.tolist()
print(f'\nColumns >99% zero ({len(ultra_sparse)}): {ultra_sparse}')

=== Zero Inflation by Column ===
(2000, 2020]                1.000000
(2020, 2030]                1.000000
N_CLMBRAND1_NEW             0.999658
N_CLMBRAND1_NEW_TO_BRAND    0.999185
BRAND1_NBRX                 0.999184
BRAND2_NBRX                 0.998987
BRAND1_NRX                  0.998912
N_CLMBRAND2_NEW_TO_BRAND    0.998890
SPK                         0.998739
N_CLMBRAND4_NEW_TO_BRAND    0.998379
COPAY                       0.997173
N_CLMBRAND2_NEW             0.997065
N_CLMBRAND1                 0.996858
BRAND1_TRX                  0.996824
BRAND2_NRX                  0.996469
SAMPLES                     0.996152
N_CLMBRAND4_NEW             0.991135
SPEC_OTHER_SPEC             0.989633
SPEC_GPFM                   0.988534
RTE                         0.988192
BRAND1_NTB_GIDX             0.986739
BRAND2_TRX                  0.986409
N_CLMBRAND2                 0.985493
ORAL_NBRX                   0.981188
BRAND1_T_GIDX               0.980778
BRAND2_NTB_GIDX             0.979531
IL23_

In [36]:
# ── Cohort bins: completely zero → drop ─────────────────────────────────────
# Columns (2000, 2020] and (2020, 2030] are 100% zero in the sample.
# If confirmed on full data, these carry zero signal.

cohort_cols = ['(1940, 1960]', '(1960, 1980]', '(1980, 2000]', '(2000, 2020]', '(2020, 2030]']
cohort_zero_rates = (df[cohort_cols] == 0).mean()
print('Cohort column zero rates:')
print(cohort_zero_rates)

# Drop cohort columns that are 100% zero
cohort_to_drop = cohort_zero_rates[cohort_zero_rates == 1.0].index.tolist()
print(f'\nDropping {len(cohort_to_drop)} fully-zero cohort cols: {cohort_to_drop}')
df = df.drop(columns=cohort_to_drop)

Cohort column zero rates:
(1940, 1960]    0.877980
(1960, 1980]    0.633797
(1980, 2000]    0.666237
(2000, 2020]    1.000000
(2020, 2030]    1.000000
dtype: float64

Dropping 2 fully-zero cohort cols: ['(2000, 2020]', '(2020, 2030]']


### 3.4 Outlier Detection (Skewed Variables)

In [37]:
# ── Identify skewed continuous variables ─────────────────────────────────────
# Rationale: Count-like variables (TRX, NRX, claims) are right-skewed.
# We flag variables with skewness > 3 for log1p transformation.
# log1p is preferred over log because it handles zeros gracefully.

# Recompute numeric columns from the current df state to avoid stale refs
current_numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
current_numeric_cols = [
    c for c in current_numeric_cols
    if c not in ['NUEVO_ID', 'YEAR', 'QTR', 'YEAR_QTR']
]

# Identify candidates: non-identifier, non-binary numeric cols
binary_like = [c for c in current_numeric_cols if df[c].isin([0, 1]).all()]
spec_geo_cols = [
    c for c in current_numeric_cols
    if c.startswith('SPEC_') or c.startswith('STATE_') or c.startswith('STS_')
]

# Columns suitable for log1p (non-negative, non-binary, non-spec/geo)
transform_candidates = [
    c for c in current_numeric_cols
    if c not in binary_like + spec_geo_cols
    and df[c].min() >= 0
]

skewness = df[transform_candidates].skew().sort_values(ascending=False)
high_skew = skewness[skewness > 3].index.tolist()

print(f'Variables with skewness > 3: {len(high_skew)}')
print(skewness[skewness > 3])

Variables with skewness > 3: 37
N_CLMBRAND1_NEW             56.205147
BRAND1_NRX                  48.056572
N_CLMBRAND1_NEW_TO_BRAND    36.917338
RTE                         36.743892
BRAND1_NBRX                 36.229274
BRAND2_NBRX                 32.031794
IL23_NRX                    31.852719
N_CLMBRAND2_NEW_TO_BRAND    31.156291
IL23_NBRX_R4_29SUM          29.408566
SPK                         28.386295
BRAND1_TRX                  27.592681
N_CLMBRAND4_NEW_TO_BRAND    26.463820
IL23_TRX                    26.262868
BRAND2_NRX                  25.318574
COPAY                       23.252086
N_CLMBRAND1                 20.225087
N_CLMBRAND2_NEW             19.640370
N_CLMOTHERS_NEW_TO_BRAND    16.252123
SAMPLES                     16.109207
IL23_NBRX                   13.404334
BRAND2_TRX                  13.169278
N_CLMBRAND4_NEW             12.619677
N_CLMBRAND2                 10.130442
N_CLMBRAND3                 10.033924
N_CLMBRAND3NEW_TO_BRAND      9.723276
N_CLMBRAND3_NEW   

In [ ]:
# ── Apply log1p transformation to high-skew variables ────────────────────────
# NOTE: We transform in-place and rename columns with '_log1p' suffix
# to maintain interpretability and allow reversal.
# EXCEPTION: We do NOT transform the primary target variables (UC_TRX, UC_NRX)
# because modeling pipelines should transform targets independently.

# Target-like columns (do not transform — leave for model pipeline)
TARGET_LIKE = ['UC_TRX', 'UC_NRX', 'ORAL_TRX', 'IL23_TRX']

# Lower-impact columns for which we intentionally skip log1p creation
LOW_IMPACT_NO_LOG1P = [
    'RTE', 'SAMPLES', 'COPAY', 'DIRECTMAIL', 'SPK', 'DETAILS',
    'N_CLMBRAND1_NEW', 'N_CLMBRAND2_NEW', 'N_CLMBRAND3_NEW',
    'N_CLMBRAND4_NEW', 'N_CLMOTHERS_NEW'
]

cols_to_transform = [
    c for c in high_skew
    if c not in TARGET_LIKE + LOW_IMPACT_NO_LOG1P
]

log1p_map = {}
for col in cols_to_transform:
    new_col = col + '_log1p'
    df[new_col] = np.log1p(df[col])
    log1p_map[col] = new_col

print(f'log1p-transformed {len(cols_to_transform)} variables:')
for orig, new in log1p_map.items():
    print(f'  {orig} → {new}')

skipped_log1p = [c for c in high_skew if c in LOW_IMPACT_NO_LOG1P]
print(f'\nSkipped log1p (lower-impact set): {skipped_log1p}')

log1p-transformed 22 variables:
  BRAND1_NRX → BRAND1_NRX_log1p
  N_CLMBRAND1_NEW_TO_BRAND → N_CLMBRAND1_NEW_TO_BRAND_log1p
  BRAND1_NBRX → BRAND1_NBRX_log1p
  BRAND2_NBRX → BRAND2_NBRX_log1p
  IL23_NRX → IL23_NRX_log1p
  N_CLMBRAND2_NEW_TO_BRAND → N_CLMBRAND2_NEW_TO_BRAND_log1p
  IL23_NBRX_R4_29SUM → IL23_NBRX_R4_29SUM_log1p
  BRAND1_TRX → BRAND1_TRX_log1p
  N_CLMBRAND4_NEW_TO_BRAND → N_CLMBRAND4_NEW_TO_BRAND_log1p
  BRAND2_NRX → BRAND2_NRX_log1p
  N_CLMBRAND1 → N_CLMBRAND1_log1p
  N_CLMOTHERS_NEW_TO_BRAND → N_CLMOTHERS_NEW_TO_BRAND_log1p
  IL23_NBRX → IL23_NBRX_log1p
  BRAND2_TRX → BRAND2_TRX_log1p
  N_CLMBRAND2 → N_CLMBRAND2_log1p
  N_CLMBRAND3 → N_CLMBRAND3_log1p
  N_CLMBRAND3NEW_TO_BRAND → N_CLMBRAND3NEW_TO_BRAND_log1p
  ORAL_NBRX → ORAL_NBRX_log1p
  N_CLMBRAND4 → N_CLMBRAND4_log1p
  UC_TRX_R4_16SUM → UC_TRX_R4_16SUM_log1p
  ORAL_NBRX_R4_29SUM → ORAL_NBRX_R4_29SUM_log1p
  N_CLMOTHERS → N_CLMOTHERS_log1p

Skipped log1p (lower-impact set): ['N_CLMBRAND1_NEW', 'RTE', 'SPK', 'COPAY', 

---
## 4. Feature Reduction
### 4.1 Remove Redundant Temporal Derivations

In [39]:
# ── YEAR, QTR, YEAR_QTR are derivable from WEEK_ID (drop to reduce schema) ───
# Rationale: WEEK_ID already encodes calendar position. We keep temporal signal
# in WEEK_ID itself and remove redundant calendar columns.

# Optional sanity check before dropping
if 'YEAR' in df.columns:
    derived_year = df['WEEK_ID'].dt.year
    year_mismatch = (df['YEAR'] != derived_year).sum()
    print(f'YEAR mismatch vs derived: {year_mismatch}')

# Drop redundant temporal columns (no YEAR_derived/QTR_derived kept)
temporal_redundant = ['YEAR', 'QTR', 'YEAR_QTR']
df = df.drop(columns=[c for c in temporal_redundant if c in df.columns])
print(f'Dropped temporal derivations: {temporal_redundant}')

YEAR mismatch vs derived: 0
Dropped temporal derivations: ['YEAR', 'QTR', 'YEAR_QTR']


### 4.2 Correlation-Based Redundancy Removal

In [40]:
# ── Correlation matrix on numeric features ───────────────────────────────────
# Threshold: |r| > 0.80 flags a pair as highly correlated.
# When two features are correlated, we KEEP the one with clearer causal
# interpretation or business meaning and DROP the derivative/proxy.

current_numeric = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude IDs
feature_cols = [c for c in current_numeric if c not in ['NUEVO_ID']]

corr_matrix = df[feature_cols].corr().abs()

# Extract pairs with high correlation
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [
    (col, row, round(upper.loc[row, col], 3))
    for col in upper.columns
    for row in upper.index
    if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.80
]

print('=== Highly Correlated Pairs (|r| > 0.80) ===')
for a, b, r in sorted(high_corr_pairs, key=lambda x: -x[2]):
    print(f'  {a} <-> {b}: {r}')

=== Highly Correlated Pairs (|r| > 0.80) ===
  N_CLMBRAND1_NEW_TO_BRAND_log1p <-> N_CLMBRAND1_NEW_TO_BRAND: 0.999
  BRAND1_NBRX_log1p <-> BRAND1_NBRX: 0.999
  BRAND2_NBRX_log1p <-> BRAND2_NBRX: 0.999
  N_CLMBRAND2_NEW_TO_BRAND_log1p <-> N_CLMBRAND2_NEW_TO_BRAND: 0.999
  N_CLMBRAND4_NEW_TO_BRAND_log1p <-> N_CLMBRAND4_NEW_TO_BRAND: 0.999
  N_CLMBRAND1_log1p <-> N_CLMBRAND1: 0.998
  BRAND2_TRX_log1p <-> BRAND2_TRX: 0.997
  BRAND1_TRX_log1p <-> BRAND1_TRX: 0.996
  N_CLMBRAND2_log1p <-> N_CLMBRAND2: 0.996
  BRAND2_NRX_log1p <-> BRAND2_NRX: 0.995
  BRAND1_NRX_log1p <-> BRAND1_NRX: 0.994
  ORAL_NBRX_log1p <-> ORAL_NBRX: 0.994
  N_CLMBRAND3NEW_TO_BRAND_log1p <-> N_CLMBRAND3NEW_TO_BRAND: 0.991
  N_CLMBRAND4_log1p <-> N_CLMBRAND4: 0.99
  IL23_NBRX_log1p <-> IL23_NBRX: 0.975
  N_CLMOTHERS_NEW_TO_BRAND_log1p <-> N_CLMOTHERS_NEW_TO_BRAND: 0.951
  N_CLMBRAND3_log1p <-> N_CLMBRAND3: 0.944
  IL23_NRX_log1p <-> IL23_NRX: 0.927
  ORAL_NBRX_R4_29SUM_log1p <-> ORAL_NBRX_R4_29SUM: 0.905
  N_CLMOTHERS_log1p

In [41]:
# ── Decision: Drop highly correlated redundant features ─────────────────────
# 
# Pair 1: YEAR_derived <-> YEAR_QTR-like → already dropped above.
#
# Pair 2: UC_TRX <-> UC_TRX_R4_16SUM (r ~ 0.84)
#   Decision: DROP UC_TRX_R4_16SUM (rolling 16-week sum)
#   Reason: If used as a feature at time T, the rolling sum includes T,
#   creating LEAKAGE if the target is UC_TRX at T. Even when lagged, the
#   same information is better captured by explicit lag features (Section 5).
#   Keep UC_TRX as it's the primary raw signal.
#
# Pair 3: BRAND1_NBRX <-> N_CLMBRAND1_NEW_TO_BRAND (r ~ 0.86)
#   Decision: KEEP BRAND1_NBRX (prescription volume)
#   DROP N_CLMBRAND1_NEW_TO_BRAND (claims-based proxy for the same event)
#   Reason: BRAND1_NBRX is the direct volume measure; the claims equivalent
#   is a noisier proxy measuring the same underlying behavior.

corr_drops = ['UC_TRX_R4_16SUM', 'N_CLMBRAND1_NEW_TO_BRAND']

# Also check log1p versions of these and drop if present
corr_drops_extended = corr_drops + [c + '_log1p' for c in corr_drops if c + '_log1p' in df.columns]
corr_drops_extended = [c for c in corr_drops_extended if c in df.columns]

df = df.drop(columns=corr_drops_extended)
print(f'Dropped (correlation-based): {corr_drops_extended}')

Dropped (correlation-based): ['UC_TRX_R4_16SUM', 'N_CLMBRAND1_NEW_TO_BRAND', 'UC_TRX_R4_16SUM_log1p', 'N_CLMBRAND1_NEW_TO_BRAND_log1p']


### 4.3 Rolling / Engineered Feature Leakage Assessment

In [42]:
# ── Evaluate rolling features for temporal leakage ──────────────────────────
# Rolling features present in dataset:
#   UC_TRX_R4_16SUM  → already dropped (correlated with target, leakage risk)
#   ORAL_NBRX_R4_29SUM → 29-week rolling sum of ORAL new-brand prescriptions
#   IL23_NBRX_R4_29SUM → 29-week rolling sum of IL23 new-brand prescriptions
#   BRAND1_NTB_GIDX, BRAND2_NTB_GIDX → NTB growth index (source: Veeva/similar)
#   BRAND1_T_GIDX, BRAND2_T_GIDX     → Total TRX growth index
#
# LEAKAGE RULE: If a rolling feature at week T includes week T in its window,
# it is a proxy of the contemporaneous target and MUST be lagged or dropped.
#
# Decision:
#   ORAL_NBRX_R4_29SUM, IL23_NBRX_R4_29SUM — KEEP with mandatory 1-period lag.
#     These capture long-horizon market dynamics that can legitimately precede
#     the target if properly lagged. We lag by 1 week (shift within panel group).
#
#   BRAND1/2_NTB_GIDX, BRAND1/2_T_GIDX — KEEP with mandatory 1-period lag.
#     Growth indices reflect market trajectory signals; lagging removes leakage.
#
# NOTE: If the exact window definition is confirmed to exclude week T,
# the lag is still recommended as a safety measure for production.

rolling_cols_to_lag = [
    'ORAL_NBRX_R4_29SUM', 'IL23_NBRX_R4_29SUM',
    'BRAND1_NTB_GIDX', 'BRAND2_NTB_GIDX',
    'BRAND1_T_GIDX', 'BRAND2_T_GIDX'
]
rolling_cols_to_lag = [c for c in rolling_cols_to_lag if c in df.columns]
print(f'Rolling/index cols that will be lagged: {rolling_cols_to_lag}')

Rolling/index cols that will be lagged: ['ORAL_NBRX_R4_29SUM', 'IL23_NBRX_R4_29SUM', 'BRAND1_NTB_GIDX', 'BRAND2_NTB_GIDX', 'BRAND1_T_GIDX', 'BRAND2_T_GIDX']


### 4.4 Claims Group Simplification

In [43]:
# ── Claims & switching feature audit ────────────────────────────────────────
# The claims group has 15 columns across 3 sub-groups:
#   Base claims:         N_CLMBRANDx, N_CLMOTHERS          (5 cols)
#   New patient claims:  N_CLMBRANDx_NEW, N_CLMOTHERS_NEW  (5 cols)
#   New-to-brand:        N_CLMBRANDx_NEW_TO_BRAND          (5 cols — but 1 dropped above)
#
# Business logic:
#   - Base claims are the total HCP-level claim volumes per brand
#   - _NEW variants track only patients who are new to any therapy in that period
#   - _NEW_TO_BRAND tracks patients switching to the brand (overlap with NBRX)
#
# Decision: DROP all _NEW_TO_BRAND claims columns.
#   Reason: New-to-brand behavior is already captured by BRAND1_NBRX / BRAND2_NBRX
#   and ORAL_NBRX / IL23_NBRX in the TRX/NRX group. Keeping both creates
#   multicollinearity without additional information for most modeling tasks.

new_to_brand_cols = [c for c in df.columns if 'NEW_TO_BRAND' in c]
print(f'NEW_TO_BRAND columns to drop: {new_to_brand_cols}')
df = df.drop(columns=new_to_brand_cols)
print(f'Shape after dropping NEW_TO_BRAND claims: {df.shape}')

NEW_TO_BRAND columns to drop: ['N_CLMBRAND3NEW_TO_BRAND', 'N_CLMBRAND4_NEW_TO_BRAND', 'N_CLMBRAND2_NEW_TO_BRAND', 'N_CLMOTHERS_NEW_TO_BRAND', 'N_CLMBRAND2_NEW_TO_BRAND_log1p', 'N_CLMBRAND4_NEW_TO_BRAND_log1p', 'N_CLMOTHERS_NEW_TO_BRAND_log1p', 'N_CLMBRAND3NEW_TO_BRAND_log1p']
Shape after dropping NEW_TO_BRAND claims: (1800066, 74)


### 4.5 Provider & Geography Mix Consolidation

In [44]:
# ── Specialty and state mix columns ─────────────────────────────────────────
# These are time-invariant or slow-changing HCP attributes measured at the
# panel level (fraction of patients by specialty/state).
#
# Decision:
#   KEEP SPEC_* columns (6 total) — speciality mix is a key segmentation driver
#   and predictive of prescribing behavior.
#
#   KEEP STATE_* columns (9 total including STS_OTHER_STS) — geography matters
#   for market access, payer mix, and KOL effects. However, we consolidate the
#   'OTHER' category (STS_OTHER_STS) which is already aggregated.
#
# No action needed — these columns are already compact.

spec_cols  = [c for c in df.columns if c.startswith('SPEC_')]
state_cols = [c for c in df.columns if c.startswith('STATE_') or c == 'STS_OTHER_STS']
print(f'Specialty mix columns ({len(spec_cols)}): {spec_cols}')
print(f'Geography mix columns ({len(state_cols)}): {state_cols}')

Specialty mix columns (6): ['SPEC_GE', 'SPEC_GPFM', 'SPEC_IM', 'SPEC_NRP', 'SPEC_OTHER_SPEC', 'SPEC_PHA']
Geography mix columns (9): ['STATE_1', 'STATE_2', 'STATE_3', 'STATE_4', 'STATE_5', 'STATE_6', 'STS_OTHER_STS', 'STATE_7', 'STATE_8']


### 4.6 Final Log1p: Drop Original Skewed Columns

In [45]:
# ── Replace originals with log1p-transformed versions ────────────────────────
# For the transformed columns, we drop the originals to avoid duplication.
# The log1p versions are the canonical features in the clean dataset.
# Target-like variables (UC_TRX, UC_NRX, etc.) are NOT transformed here.

originals_to_drop = [orig for orig, new in log1p_map.items() if orig in df.columns and new in df.columns]
df = df.drop(columns=originals_to_drop)
print(f'Dropped {len(originals_to_drop)} original (pre-transform) columns:')
print(originals_to_drop)

Dropped 16 original (pre-transform) columns:
['BRAND1_NRX', 'BRAND1_NBRX', 'BRAND2_NBRX', 'IL23_NRX', 'IL23_NBRX_R4_29SUM', 'BRAND1_TRX', 'BRAND2_NRX', 'N_CLMBRAND1', 'IL23_NBRX', 'BRAND2_TRX', 'N_CLMBRAND2', 'N_CLMBRAND3', 'ORAL_NBRX', 'N_CLMBRAND4', 'ORAL_NBRX_R4_29SUM', 'N_CLMOTHERS']


---
## 5. Temporal Integrity & Lag Feature Construction

In [46]:
# ── Sort panel by ID and time (mandatory before any lag operation) ───────────
df = df.sort_values(['NUEVO_ID', 'WEEK_ID']).reset_index(drop=True)
print('Sorted by NUEVO_ID, WEEK_ID ✓')

Sorted by NUEVO_ID, WEEK_ID ✓


In [47]:
# ── Create 1-week lag features for rolling and target-like variables ─────────
# Rationale: Any feature that reflects activity at week T risks leaking
# information if used to predict T. Lagging by 1 week (within each ID)
# ensures features only contain information available BEFORE the target week.
#
# Lag strategy:
#   1. Rolling features (ORAL_NBRX_R4_29SUM, IL23_NBRX_R4_29SUM, etc.) → lag 1
#   2. Promotion variables are represented in the final dataset ONLY as lagged
#      versions (no contemporaneous promo columns kept) to support causal setup.
#   3. TRX/NRX — create L1 versions as lagged features; originals remain as targets.

# Columns to lag
LAG1_FEATURES = rolling_cols_to_lag + ['UC_TRX', 'UC_NRX']
LAG1_FEATURES = [c for c in LAG1_FEATURES if c in df.columns]

for col in LAG1_FEATURES:
    lag_col = col + '_L1'
    df[lag_col] = df.groupby('NUEVO_ID')[col].shift(1)

print(f'Created L1 lag features for: {LAG1_FEATURES}')

# After lagging, the first week for each ID will have NaN in lag features.
lag_null_count = df[[c + '_L1' for c in LAG1_FEATURES]].isnull().sum()
print(f'\nNaN in lag features (first-week rows per ID):')
print(lag_null_count)

Created L1 lag features for: ['BRAND1_NTB_GIDX', 'BRAND2_NTB_GIDX', 'BRAND1_T_GIDX', 'BRAND2_T_GIDX', 'UC_TRX', 'UC_NRX']

NaN in lag features (first-week rows per ID):
BRAND1_NTB_GIDX_L1    20931
BRAND2_NTB_GIDX_L1    20931
BRAND1_T_GIDX_L1      20931
BRAND2_T_GIDX_L1      20931
UC_TRX_L1             20931
UC_NRX_L1             20931
dtype: int64


In [48]:
# ── Create L1 lag for promotion variables (optional, flagged for use) ────────
# These are available both at time T (for promotion mix analysis)
# and lagged (for causal TRX prediction). We create both.

# Prefer raw promo columns; if they were replaced by log1p in Section 4.6,
# fall back to the *_log1p versions so lag features are still created.
PROMO_BASE = ['RTE', 'SAMPLES', 'COPAY', 'DIRECTMAIL', 'SPK', 'DETAILS']

promo_col_map = {}
PROMO_COLS = []
for base_col in PROMO_BASE:
    if base_col in df.columns:
        chosen_col = base_col
    elif f'{base_col}_log1p' in df.columns:
        chosen_col = f'{base_col}_log1p'
    else:
        continue

    PROMO_COLS.append(chosen_col)
    promo_col_map[base_col] = chosen_col

for col in PROMO_COLS:
    df[col + '_L1'] = df.groupby('NUEVO_ID')[col].shift(1)

print('Promotion lag column mapping (base → used):')
print(promo_col_map)
print(f'Created L1 lag for promotion variables: {PROMO_COLS}')

Promotion lag column mapping (base → used):
{'RTE': 'RTE', 'SAMPLES': 'SAMPLES', 'COPAY': 'COPAY', 'DIRECTMAIL': 'DIRECTMAIL', 'SPK': 'SPK', 'DETAILS': 'DETAILS'}
Created L1 lag for promotion variables: ['RTE', 'SAMPLES', 'COPAY', 'DIRECTMAIL', 'SPK', 'DETAILS']


---
## 6. Final Dataset Construction & Export

In [49]:
# ── Define final column order (logical grouping) ─────────────────────────────

def get_cols(prefix_or_list):
    if isinstance(prefix_or_list, list):
        return [c for c in prefix_or_list if c in df.columns]
    return [c for c in df.columns if c.startswith(prefix_or_list)]

final_cols = (
    # Identifiers
    ['NUEVO_ID', 'WEEK_ID'] +
    # Segmentation
    ['ATSEG', 'ATSEG_IS_MISSING'] +
    # Target-like (raw, for modeling)
    [c for c in ['UC_TRX', 'UC_NRX', 'ORAL_TRX', 'IL23_TRX'] if c in df.columns] +
    # Lagged targets
    [c for c in df.columns if c.endswith('_L1') and any(t in c for t in ['UC_TRX','UC_NRX'])] +
    # Brand volume (TRX/NRX/NBRX)
    [c for c in df.columns if any(c.startswith(p) for p in ['BRAND1_','BRAND2_','ORAL_N','IL23_N']) and not c.endswith('_L1')] +
    # Claims (base + new patient)
    [c for c in df.columns if c.startswith('N_CLM') and '_NEW_TO_BRAND' not in c] +
    # Promotion (lagged only — no contemporaneous promo columns)
    [c + '_L1' for c in PROMO_COLS if c + '_L1' in df.columns] +
    # Rolling features (lagged versions only)
    [c for c in df.columns if ('R4_' in c or '_GIDX' in c) and c.endswith('_L1')] +
    # Provider mix
    get_cols('SPEC_') +
    # Geography mix
    state_cols +
    # Cohort mix (remaining)
    [c for c in df.columns if c.startswith('(') and c in df.columns]
)

# Deduplicate while preserving order
seen = set()
final_cols = [c for c in final_cols if c in df.columns and not (c in seen or seen.add(c))]

# Catch any remaining columns not yet placed
excluded_from_append = set(PROMO_COLS)
remaining = [c for c in df.columns if c not in final_cols and c not in excluded_from_append]
if remaining:
    print(f'Columns not in ordered list (will be appended): {remaining}')
    final_cols += remaining

df_clean = df[final_cols].copy()
print(f'Final dataset shape: {df_clean.shape}')

Final dataset shape: (1800066, 64)


In [50]:
# ── Final sanity checks ──────────────────────────────────────────────────────

print('=== FINAL SANITY CHECKS ===')

# 1. No duplicate ID×WEEK
dupes = df_clean.duplicated(subset=['NUEVO_ID','WEEK_ID']).sum()
print(f'  Duplicate rows: {dupes}  {"✓" if dupes==0 else "✗ FIX REQUIRED"}')

# 2. WEEK_ID is datetime
print(f'  WEEK_ID dtype: {df_clean["WEEK_ID"].dtype}  {"✓" if str(df_clean["WEEK_ID"].dtype).startswith("datetime") else "✗"}')

# 3. No columns with 100% nulls
all_null = df_clean.columns[df_clean.isnull().all()].tolist()
print(f'  All-null columns: {all_null if all_null else "None ✓"}')

# 4. ATSEG_IS_MISSING is binary
flag_vals = df_clean['ATSEG_IS_MISSING'].unique()
print(f'  ATSEG_IS_MISSING values: {sorted(flag_vals)}  {"✓" if set(flag_vals) <= {0,1} else "✗"}')

# 5. No negative values in log1p columns (they should all be >= 0)
log1p_cols_present = [c for c in df_clean.columns if c.endswith('_log1p')]
neg_log1p = {c: (df_clean[c] < 0).sum() for c in log1p_cols_present if (df_clean[c] < 0).sum() > 0}
print(f'  Negative values in log1p cols: {neg_log1p if neg_log1p else "None ✓"}')

print()
print(f'FINAL SHAPE: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')

=== FINAL SANITY CHECKS ===
  Duplicate rows: 0  ✓
  WEEK_ID dtype: datetime64[ns]  ✓
  All-null columns: None ✓
  ATSEG_IS_MISSING values: [0, 1]  ✓
  Negative values in log1p cols: None ✓

FINAL SHAPE: 1,800,066 rows × 64 columns


In [51]:
# ── Export clean dataset ─────────────────────────────────────────────────────
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f'Exported: {OUTPUT_FILE}')
print(f'File ready for modeling pipeline.')

Exported: VELSIPITY_cleaned_igna.csv
File ready for modeling pipeline.


---
## 7. Summary Report

In [52]:
# ── Before vs After ──────────────────────────────────────────────────────────
print('=' * 55)
print('  CLEANING & REDUCTION SUMMARY')
print('=' * 55)

raw_cols_set = set(df_raw.columns)
clean_cols_set = set(df_clean.columns)
n_removed = len(raw_cols_set - clean_cols_set)
n_added = len(clean_cols_set - raw_cols_set)
net_change = len(clean_cols_set) - len(raw_cols_set)

print(f'  Rows     : {df_raw.shape[0]:>10,}  →  {df_clean.shape[0]:,}  (unchanged)')
print(f'  Columns  : {df_raw.shape[1]:>10}  →  {df_clean.shape[1]}')
print(f'  Schema Δ : +{n_added} added, -{n_removed} removed, net {net_change:+d}')
print()
print('  Features REMOVED:')
print(f'    - Temporal derivations (YEAR, QTR, YEAR_QTR)            : 3')
print(f'    - Fully-zero cohort bins                                  : {len(cohort_to_drop)}')
print(f'    - Correlation-based drops                                 : {len(corr_drops_extended)}')
print(f'    - NEW_TO_BRAND claims (redundant with NBRX)               : {len(new_to_brand_cols)}')
print(f'    - Original skewed cols replaced by log1p                  : {len(originals_to_drop)}')
print()
print('  Features ADDED:')
print(f'    - log1p transforms                                        : {len(log1p_map)}')
print(f'    - ATSEG_IS_MISSING flag                                   : 1')
print(f'    - Lag features (L1) for rolling + target + promo          : {len(LAG1_FEATURES) + len(PROMO_COLS)}')
print()
print('  Final schema groups:')
print(f'    Identifiers/temporal : {len([c for c in df_clean.columns if c in ["NUEVO_ID","WEEK_ID"]])}')
print(f'    Segmentation         : {len([c for c in df_clean.columns if "ATSEG" in c])}')
print(f'    TRX/NRX/NBRX targets : {len([c for c in df_clean.columns if any(x in c for x in ["TRX","NRX","NBRX"]) and not c.endswith("_L1")])}')
print(f'    Lag features (L1)    : {len([c for c in df_clean.columns if c.endswith("_L1")])}')
print(f'    Claims features      : {len([c for c in df_clean.columns if c.startswith("N_CLM")])}')
print(f'    Promotion features   : {len([c for c in df_clean.columns if c in PROMO_COLS])}')
print(f'    Provider mix (SPEC)  : {len([c for c in df_clean.columns if c.startswith("SPEC_")])}')
print(f'    Geography mix (STATE): {len(state_cols)}')
print(f'    Log1p transforms     : {len([c for c in df_clean.columns if c.endswith("_log1p")])}')
print('=' * 55)

  CLEANING & REDUCTION SUMMARY
  Rows     :  1,800,066  →  1,800,066  (unchanged)
  Columns  :         68  →  64
  Schema Δ : +29 added, -33 removed, net -4

  Features REMOVED:
    - Temporal derivations (YEAR, QTR, YEAR_QTR)            : 3
    - Fully-zero cohort bins                                  : 2
    - Correlation-based drops                                 : 4
    - NEW_TO_BRAND claims (redundant with NBRX)               : 8
    - Original skewed cols replaced by log1p                  : 16

  Features ADDED:
    - log1p transforms                                        : 22
    - ATSEG_IS_MISSING flag                                   : 1
    - Lag features (L1) for rolling + target + promo          : 12

  Final schema groups:
    Identifiers/temporal : 2
    Segmentation         : 2
    TRX/NRX/NBRX targets : 16
    Lag features (L1)    : 12
    Claims features      : 10
    Promotion features   : 0
    Provider mix (SPEC)  : 6
    Geography mix (STATE): 9
    Log1p trans

In [53]:
# ── Schema delta audit (raw vs clean) ────────────────────────────────────────
raw_cols = set(df_raw.columns)
clean_cols = set(df_clean.columns)

added_cols = sorted(clean_cols - raw_cols)
removed_cols = sorted(raw_cols - clean_cols)

print('=== SCHEMA DELTA AUDIT ===')
print(f'Raw columns   : {len(raw_cols)}')
print(f'Clean columns : {len(clean_cols)}')
print(f'Net change    : {len(clean_cols) - len(raw_cols):+d}')
print(f'Added columns : {len(added_cols)}')
print(f'Removed cols  : {len(removed_cols)}')
print()
print('Added columns (first 30):')
print(added_cols[:30])
print()
print('Removed columns (first 30):')
print(removed_cols[:30])

=== SCHEMA DELTA AUDIT ===
Raw columns   : 68
Clean columns : 64
Net change    : -4
Added columns : 29
Removed cols  : 33

Added columns (first 30):
['ATSEG_IS_MISSING', 'BRAND1_NBRX_log1p', 'BRAND1_NRX_log1p', 'BRAND1_NTB_GIDX_L1', 'BRAND1_TRX_log1p', 'BRAND1_T_GIDX_L1', 'BRAND2_NBRX_log1p', 'BRAND2_NRX_log1p', 'BRAND2_NTB_GIDX_L1', 'BRAND2_TRX_log1p', 'BRAND2_T_GIDX_L1', 'COPAY_L1', 'DETAILS_L1', 'DIRECTMAIL_L1', 'IL23_NBRX_R4_29SUM_log1p', 'IL23_NBRX_log1p', 'IL23_NRX_log1p', 'N_CLMBRAND1_log1p', 'N_CLMBRAND2_log1p', 'N_CLMBRAND3_log1p', 'N_CLMBRAND4_log1p', 'N_CLMOTHERS_log1p', 'ORAL_NBRX_R4_29SUM_log1p', 'ORAL_NBRX_log1p', 'RTE_L1', 'SAMPLES_L1', 'SPK_L1', 'UC_NRX_L1', 'UC_TRX_L1']

Removed columns (first 30):
['(2000, 2020]', '(2020, 2030]', 'BRAND1_NBRX', 'BRAND1_NRX', 'BRAND1_TRX', 'BRAND2_NBRX', 'BRAND2_NRX', 'BRAND2_TRX', 'COPAY', 'DETAILS', 'DIRECTMAIL', 'IL23_NBRX', 'IL23_NBRX_R4_29SUM', 'IL23_NRX', 'N_CLMBRAND1', 'N_CLMBRAND1_NEW_TO_BRAND', 'N_CLMBRAND2', 'N_CLMBRAND2_NEW_

In [54]:
# ── Final column list ────────────────────────────────────────────────────────
print('Final columns:')
for i, col in enumerate(df_clean.columns, 1):
    dtype = str(df_clean[col].dtype)
    null_pct = df_clean[col].isnull().mean()
    print(f'  {i:3}. {col:<40} {dtype:<12} null={null_pct:.1%}')

Final columns:
    1. NUEVO_ID                                 int64        null=0.0%
    2. WEEK_ID                                  datetime64[ns] null=0.0%
    3. ATSEG                                    object       null=43.2%
    4. ATSEG_IS_MISSING                         int32        null=0.0%
    5. UC_TRX                                   float64      null=0.0%
    6. UC_NRX                                   float64      null=0.0%
    7. ORAL_TRX                                 float64      null=0.0%
    8. IL23_TRX                                 float64      null=0.0%
    9. UC_TRX_L1                                float64      null=1.2%
   10. UC_NRX_L1                                float64      null=1.2%
   11. ORAL_NRX                                 float64      null=0.0%
   12. BRAND1_NTB_GIDX                          float64      null=0.0%
   13. BRAND2_NTB_GIDX                          float64      null=0.0%
   14. BRAND1_T_GIDX                            float64    